# Gold — run every *_scd1 dimension (task `gold_dims_scd1`)

Runs the per-table SCD 1 notebooks (poc checksum method, helper `_scd`) in dependency order with `dbutils.notebook.run`, so the job needs one task and one compute start. Test tables only: nothing in the job reads them; the live dims (`dims.ipynb`, `restaurants.ipynb`) are unchanged.

| Order | Notebook | Needs |
|---|---|---|
| 1 | dim_channel_scd1 | stg_channels |
| 2 | dim_video_scd1 | stg_videos, stg_video_classification |
| 3 | dim_commenter_scd1 | stg_comments |
| 4 | dim_keyword_scd1 | stg_search_results |
| 5 | dim_comment_scd1 | 2 + 3 |
| 6 | dim_google_review_scd1 | dim_restaurant (task gold_restaurants) |

Not converted: `dim_date` (generated calendar, SCD 0), `dim_restaurant` (SCD 2), `dim_dish` (its source, the restaurant-keyed mentions, is built inside `restaurants.ipynb`; it needs that logic moved out first).

A table that does not exist yet is loaded with `load_type = full` first (like poc `update_load_type`). A failing table does not stop the others; the task fails at the end if any failed.

In [ ]:
for w, default in (("silver", "prod.silver"), ("gold", "prod.gold"), ("load_type", "incremental"), ("from_date", ""), ("to_date", "")):
    dbutils.widgets.text(w, default)
args = {w: dbutils.widgets.get(w) for w in ("silver", "gold", "load_type", "from_date", "to_date")}
ORDER = ["dim_channel_scd1", "dim_video_scd1", "dim_commenter_scd1", "dim_keyword_scd1", "dim_comment_scd1", "dim_google_review_scd1"]
failed = {}
for name in ORDER:
    try:
        # first load of a table = full (poc: load_type full once, then update_load_type switches it to incremental)
        run_args = dict(args, load_type="full") if not spark.catalog.tableExists(f"{args['gold']}.{name}") else args
        out = dbutils.notebook.run(f"./{name}", 1800, run_args)
        print(f"✅ {name} ({run_args['load_type']})" + (f": {out}" if out else ""))
    except Exception as e:
        failed[name] = str(e)[:500]
        print(f"❌ {name}: {str(e)[:500]}")
if failed:
    raise RuntimeError(f"{len(failed)} of {len(ORDER)} SCD 1 dims failed: {list(failed)}")
print(f"--- {len(ORDER)} SCD 1 dims done ({args['load_type']}) ---")